# Verify Claim

Computational verification of quantitative claims: loads cited data sources,
recomputes claimed values, checks distributions, cross-references, and flags discrepancies.

**Outputs:** `verification-{date}.json`, `verification-summary-{date}.md`

In [ ]:
# Papermill parameters
# claims_json: JSON string — list of {statement, cited_value, cited_source, metric_name}
claims_json = '[]'
# data_sources: comma-separated file paths or "bigquery" for BQ verification
data_sources = ""
report_date = "2026-04-04"
bq_project = None
output_dir = None

In [ ]:
import sys
import json
from pathlib import Path

# Find workspace root (walk up to .git), then add data/notebooks/ to sys.path
# so that utils.pm_helpers is importable regardless of cwd
# (papermill runs notebooks from outputs/, not templates/)
_dir = Path.cwd()
while _dir != _dir.parent:
    if (_dir / ".git").exists():
        break
    _dir = _dir.parent
WORKSPACE_ROOT = _dir
NOTEBOOKS_DIR = WORKSPACE_ROOT / "data" / "notebooks"
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

import pandas as pd
import numpy as np
from utils.pm_helpers import (
    load_csv,
    load_csv_reports,
    set_chart_style,
    save_json,
    BRAND_COLOURS,
)

set_chart_style()

# Parse claims
claims = json.loads(claims_json) if isinstance(claims_json, str) else claims_json
source_paths = [s.strip() for s in data_sources.split(",") if s.strip()] if data_sources else []

print(f"Claims to verify: {len(claims)}")
print(f"Data sources: {source_paths or 'none specified'}")

In [ ]:
# --- Load data sources ---

loaded_sources = {}

for src in source_paths:
    if src.lower() == "bigquery":
        loaded_sources["bigquery"] = "available"
        continue
    p = Path(src) if Path(src).is_absolute() else WORKSPACE_ROOT / src
    if p.exists() and p.suffix == ".csv":
        try:
            df = pd.read_csv(p)
            df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
            loaded_sources[src] = df
            print(f"Loaded: {src} ({len(df)} rows, {len(df.columns)} cols)")
        except Exception as e:
            print(f"Failed to load {src}: {e}")
    else:
        print(f"Not found: {src}")

print(f"\n{len(loaded_sources)} sources loaded.")

In [ ]:
# --- Verify each claim ---

verification_results = []

def find_metric_in_sources(metric_name, sources):
    """Search loaded DataFrames for a column matching the metric name."""
    metric_lower = metric_name.lower().replace(" ", "_")
    for src_name, src_data in sources.items():
        if isinstance(src_data, pd.DataFrame):
            for col in src_data.columns:
                if metric_lower in col.lower() or col.lower() in metric_lower:
                    return src_name, src_data, col
    return None, None, None

def parse_numeric(val):
    """Parse a numeric value from various string formats."""
    if isinstance(val, (int, float)):
        return float(val)
    if isinstance(val, str):
        cleaned = val.replace(",", "").replace("£", "").replace("$", "").replace("%", "").strip()
        try:
            return float(cleaned)
        except ValueError:
            return None
    return None

for i, claim in enumerate(claims):
    result = {
        "claim_index": i,
        "statement": claim.get("statement", ""),
        "cited_value": claim.get("cited_value"),
        "cited_source": claim.get("cited_source", ""),
        "metric_name": claim.get("metric_name", ""),
        "status": "unverified",
    }

    cited_num = parse_numeric(claim.get("cited_value"))
    if cited_num is None:
        result["status"] = "unparseable"
        result["note"] = "Could not parse cited value as numeric"
        verification_results.append(result)
        continue

    # Find metric in loaded sources
    src_name, src_df, col_name = find_metric_in_sources(claim.get("metric_name", ""), loaded_sources)

    if src_df is None:
        result["status"] = "source_not_found"
        result["note"] = f"Could not find metric '{claim.get('metric_name')}' in loaded sources"
        verification_results.append(result)
        continue

    # Recompute from raw data
    series = pd.to_numeric(src_df[col_name], errors="coerce").dropna()
    if len(series) == 0:
        result["status"] = "no_data"
        result["note"] = f"Column '{col_name}' in '{src_name}' has no numeric values"
        verification_results.append(result)
        continue

    # Compute various aggregations to find a match
    computed = {
        "last_value": float(series.iloc[-1]),
        "mean": float(series.mean()),
        "median": float(series.median()),
        "sum": float(series.sum()),
        "max": float(series.max()),
        "min": float(series.min()),
    }
    result["computed_values"] = computed
    result["source_used"] = src_name
    result["column_used"] = col_name
    result["n_observations"] = len(series)

    # Check if cited value matches any computed value (within 1% tolerance)
    tolerance = 0.01
    matched = False
    for agg_name, agg_val in computed.items():
        if agg_val != 0 and abs(cited_num - agg_val) / abs(agg_val) <= tolerance:
            result["status"] = "verified"
            result["matched_aggregation"] = agg_name
            result["computed_value"] = agg_val
            result["discrepancy_pct"] = round((cited_num - agg_val) / abs(agg_val) * 100, 4)
            matched = True
            break

    if not matched:
        # Find closest match
        closest_agg = min(computed.items(), key=lambda x: abs(x[1] - cited_num) if x[1] != 0 else float("inf"))
        result["status"] = "discrepancy"
        result["closest_aggregation"] = closest_agg[0]
        result["closest_computed_value"] = closest_agg[1]
        if closest_agg[1] != 0:
            result["discrepancy_pct"] = round((cited_num - closest_agg[1]) / abs(closest_agg[1]) * 100, 2)

    # Distribution analysis
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    result["distribution"] = {
        "mean": round(float(series.mean()), 4),
        "median": round(float(series.median()), 4),
        "std": round(float(series.std()), 4),
        "q1": round(float(q1), 4),
        "q3": round(float(q3), 4),
        "iqr": round(float(iqr), 4),
        "outlier_lower": round(float(q1 - 1.5 * iqr), 4),
        "outlier_upper": round(float(q3 + 1.5 * iqr), 4),
        "n_outliers": int(((series < q1 - 1.5 * iqr) | (series > q3 + 1.5 * iqr)).sum()),
        "mean_vs_median_divergence": "significant" if abs(series.mean() - series.median()) > 0.1 * series.std() else "minor",
    }

    verification_results.append(result)
    status_icon = "VERIFIED" if result["status"] == "verified" else "DISCREPANCY" if result["status"] == "discrepancy" else result["status"].upper()
    print(f"Claim {i}: {status_icon} — \"{claim.get('statement', '')[:60]}\"")

print(f"\n=== Verification Summary ===")
for status in ["verified", "discrepancy", "source_not_found", "unparseable", "no_data", "unverified"]:
    count = sum(1 for r in verification_results if r["status"] == status)
    if count > 0:
        print(f"  {status}: {count}")

In [ ]:
# --- Cross-reference checks ---

cross_checks = []

# Check arithmetic consistency: conversions - churned = paid_conversions
for src_name, src_data in loaded_sources.items():
    if not isinstance(src_data, pd.DataFrame):
        continue
    cols = src_data.columns.str.lower()

    gs_col = next((c for c in src_data.columns if "conversions" in c.lower()), None)
    cancel_col = next((c for c in src_data.columns if "cancellation" in c.lower()), None)
    ns_col = next((c for c in src_data.columns if "paid_conversions" in c.lower()), None)

    if gs_col and cancel_col and ns_col:
        gs = pd.to_numeric(src_data[gs_col], errors="coerce")
        cancel = pd.to_numeric(src_data[cancel_col], errors="coerce")
        ns = pd.to_numeric(src_data[ns_col], errors="coerce")
        computed_ns = gs - cancel
        diff = (ns - computed_ns).dropna()
        if len(diff) > 0:
            max_diff = diff.abs().max()
            check = {
                "check": "conversions - churned = paid_conversions",
                "source": src_name,
                "max_discrepancy": float(max_diff),
                "status": "consistent" if max_diff < 1 else "inconsistent",
            }
            cross_checks.append(check)
            print(f"Cross-check ({src_name}): GS - Cancellations = NS — {check['status']} (max diff: {max_diff:.0f})")

if not cross_checks:
    print("No cross-reference checks possible with available data.")

In [ ]:
# --- Export ---

output = {
    "report_date": report_date,
    "total_claims": len(claims),
    "verified": sum(1 for r in verification_results if r["status"] == "verified"),
    "discrepancies": sum(1 for r in verification_results if r["status"] == "discrepancy"),
    "unverifiable": sum(1 for r in verification_results if r["status"] not in ("verified", "discrepancy")),
    "results": verification_results,
    "cross_checks": cross_checks,
}
save_json(output, "verification", report_date, output_dir)

# Summary markdown
lines = [f"# Claim Verification — {report_date}\n"]
lines.append(f"**Claims checked:** {output['total_claims']}")
lines.append(f"**Verified:** {output['verified']} | **Discrepancies:** {output['discrepancies']} | **Unverifiable:** {output['unverifiable']}\n")

lines.append("| # | Claim | Cited | Computed | Status |")
lines.append("|---|-------|-------|----------|--------|")
for r in verification_results:
    cited = r.get("cited_value", "—")
    computed = r.get("computed_value", r.get("closest_computed_value", "—"))
    status = r["status"].upper()
    stmt = r["statement"][:50] + "..." if len(r["statement"]) > 50 else r["statement"]
    lines.append(f"| {r['claim_index']} | {stmt} | {cited} | {computed} | {status} |")

if cross_checks:
    lines.append("\n## Cross-reference Checks\n")
    for cc in cross_checks:
        lines.append(f"- **{cc['check']}** ({cc['source']}): {cc['status']}")

summary_md = "\n".join(lines)
md_path = Path(output_dir or str(WORKSPACE_ROOT / "data" / "reports" / "charts")) / f"verification-summary-{report_date}.md"
md_path.parent.mkdir(parents=True, exist_ok=True)
md_path.write_text(summary_md)
print(f"\nSaved verification summary: {md_path}")